In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install pandas

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv("ai_mock_interview_labeled_dataset_30000_rows.csv")

In [ ]:
df["text"] = "Question: " + df["question"] + " Answer: " + df["answer"]

In [ ]:
df = df.drop_duplicates(subset=["text"])
print(df.shape)

In [ ]:
#from sklearn.model_selection import train_test_split

#train_texts, val_texts, train_labels, val_labels = train_test_split(
 #  df["label"].tolist(),
  #  test_size=0.2,
   # random_state=42,
    #stratify=df["label"]
#)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(
    gss.split(df["text"], df["label"], groups=df["question_id"])
)

In [ ]:
train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]

In [ ]:
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()

val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()

In [ ]:
#Check data leakage
print(len(set(train_texts).intersection(set(val_texts))))

In [ ]:
print(len(train_texts))
print(len(val_texts))
print(len(train_labels))
print(len(val_labels))

In [ ]:
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
#tokenize training Data
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
#tokenize validation data
val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
import torch

class InterviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)



In [ ]:
train_dataset = InterviewDataset(train_encodings, train_labels)
val_dataset = InterviewDataset(val_encodings, val_labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100,
    save_total_limit=2
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
preds = trainer.predict(val_dataset)

y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:\n", classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
import random

def shuffle_words(text):
    words = text.split()
    random.shuffle(words)
    return " ".join(words)

In [ ]:
shuffled_val_texts = [shuffle_words(t) for t in val_texts]

In [ ]:
shuffled_encodings = tokenizer(
    shuffled_val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
shuffled_dataset = InterviewDataset(shuffled_encodings, val_labels)

In [ ]:
preds = trainer.predict(shuffled_dataset)
y_pred = preds.predictions.argmax(-1)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy on shuffled text:", accuracy_score(val_labels, y_pred))

In [ ]:
model.save_pretrained("mock_interview_model")
tokenizer.save_pretrained("mock_interview_model")

In [ ]:
import torch

def evaluate_answer(question, answer):

    text = "Question: " + question + " Answer: " + answer

    # tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # move inputs to same device as model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # inference
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return pred

In [ ]:
question = "Explain machine learning"

answer1 = "Machine learning allows computers to learn patterns from data and make predictions."

answer2 = "I don't know anything about this."

print(evaluate_answer(question, answer1))
print(evaluate_answer(question, answer2))

In [ ]:
def generate_feedback(score):

    if score == 0:
        return "Your answer is weak. Try explaining the concept clearly."

    elif score == 1:
        return "Your answer is okay but needs more details or examples."

    else:
        return "Excellent answer with clear explanation."

In [ ]:
score = evaluate_answer(question, answer1)

print("Score:", score)
print("Feedback:", generate_feedback(score))